# CT3 Four-Iteration 199-Case Baseline

## Goal

Train interpretable and nonlinear baseline models using the same frozen
case splits, the same complete elements, the same selected feature set,
and the same upper-tail weights that will be used by symbolic regression.

The baseline does not create the symbolic formula.  It establishes how
much predictive accuracy is available from the current inputs and gives
PySR an honest comparison target.  The final 50 cases remain locked.


## 1. Setup and locked inputs


In [ ]:
from pathlib import Path
import gc
import json
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
PACKAGE_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'src' / 'ct3_common.py').is_file()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory')
sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from ct3_common import *

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
warnings.filterwarnings("ignore")

CASE_DIR_OVERRIDE = None
PATHS = build_paths(PACKAGE_ROOT, CASE_DIR_OVERRIDE)
CASE_FILES, CASE_INVENTORY = discover_case_files(PATHS.case_dir)
CASE_PATH_BY_ID = case_path_lookup(CASE_INVENTORY)
SPLIT_MANIFEST = load_frozen_manifest(PATHS.manifest_path, CASE_INVENTORY)
DEVELOPMENT_CASE_IDS = development_case_ids(SPLIT_MANIFEST)
FINAL_CASE_IDS = final_test_case_ids(SPLIT_MANIFEST)

print("Package root:", PATHS.package_root)
print("Case directory:", PATHS.case_dir)
print("Cases:", len(CASE_INVENTORY))
print("Development cases:", len(DEVELOPMENT_CASE_IDS))
print("Locked final-test cases:", len(FINAL_CASE_IDS))
print("Random seed:", RANDOM_SEED)

try:
    from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
except Exception as exc:
    raise ImportError("scikit-learn is required for baseline training") from exc

OUTPUT_DIR = PATHS.output_root / "01_baseline_199cases"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_SELECTION_PATH = PATHS.output_root / "00_qc_sensitivity_ablation" / "locked_feature_set.csv"
FEATURE_SET_NAME, MODEL_FEATURES = feature_set_from_selection(FEATURE_SELECTION_PATH)
RUN_BASELINE_TRAINING = True

print("Locked feature set:", FEATURE_SET_NAME)
print("Features:", MODEL_FEATURES)


## 2. Model definitions

Ridge is a linear reference.  HistGradientBoosting tests smooth nonlinear
partitioning.  ExtraTrees tests flexible interactions and local spatial
structure.  All three use the same case-level split and upper-tail sample
weights, so their metrics are directly comparable to the weighted PySR
objective.  Model selection is validation-only.


In [ ]:
def make_models():
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            loss="squared_error", learning_rate=0.05, max_iter=180,
            max_leaf_nodes=63, min_samples_leaf=256, l2_regularization=1.0,
            early_stopping=False, random_state=RANDOM_SEED,
        ),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=64, max_depth=24, min_samples_leaf=256,
            max_features=1.0, bootstrap=False, n_jobs=4, # -1 / 4
            random_state=RANDOM_SEED,
        ),
    }

model_parameters = pd.DataFrame([
    {"model": name, "parameters_json": json.dumps(model.get_params(), default=str, sort_keys=True)}
    for name, model in make_models().items()
])
model_parameters.to_csv(OUTPUT_DIR / "baseline_model_parameters.csv", index=False)
display(model_parameters)


## 3. Run four complete case-level iterations


In [ ]:
def evaluate_model_cases(model, case_ids, iteration, model_name, split):
    rows = []
    for case_id in case_ids:
        payload, _ = load_case_payload(case_id, CASE_PATH_BY_ID, MODEL_FEATURES)
        predicted = model.predict(payload["X"])
        rows.append({
            "iteration": iteration, "model": model_name, "split": split,
            "case_id": case_id, **evaluate_prediction_arrays(payload["y"], predicted),
        })
        del payload, predicted
        gc.collect()
    return rows

case_metric_records = []
timing_records = []

if RUN_BASELINE_TRAINING:
    for iteration in range(1, 5):
        train_ids = cases_for_role(SPLIT_MANIFEST, iteration, "train")
        validation_ids = cases_for_role(SPLIT_MANIFEST, iteration, "validation")
        internal_ids = cases_for_role(SPLIT_MANIFEST, iteration, "internal_test")
        assert_no_final_cases(train_ids + validation_ids + internal_ids, SPLIT_MANIFEST)

        assembly_start = time.perf_counter()
        bundle = assemble_full_training_arrays(
            train_ids, CASE_PATH_BY_ID, MODEL_FEATURES,
            scale=False, use_tail_weights=True,
        )
        assembly_seconds = time.perf_counter() - assembly_start
        bundle["audit"].assign(iteration=iteration).to_csv(
            OUTPUT_DIR / f"training_weight_audit_iteration_{iteration}.csv", index=False
        )
        print(f"Iteration {iteration}: {len(train_ids)} cases / {len(bundle['y'])} elements")

        for model_name, model in make_models().items():
            fit_start = time.perf_counter()
            if model_name == "Ridge":
                model.fit(bundle["X"], bundle["y"], ridge__sample_weight=bundle["weights"])
            else:
                model.fit(bundle["X"], bundle["y"], sample_weight=bundle["weights"])
            fit_seconds = time.perf_counter() - fit_start

            for split_name, ids in {
                "train": train_ids,
                "validation": validation_ids,
                "internal_test": internal_ids,
            }.items():
                case_metric_records.extend(evaluate_model_cases(
                    model, ids, iteration, model_name, split_name
                ))
            timing_records.append({
                "iteration": iteration, "model": model_name,
                "n_train_cases": len(train_ids), "n_train_elements": len(bundle["y"]),
                "assembly_seconds_shared_by_iteration": assembly_seconds,
                "fit_seconds": fit_seconds,
            })
            pd.DataFrame(case_metric_records).to_csv(
                OUTPUT_DIR / "baseline_case_metrics.csv", index=False
            )
            pd.DataFrame(timing_records).to_csv(
                OUTPUT_DIR / "baseline_timing.csv", index=False
            )
            del model
            gc.collect()
        del bundle
        gc.collect()

    case_metrics = pd.DataFrame(case_metric_records)
    split_metrics = aggregate_case_metrics(case_metrics, ["iteration", "model", "split"])
    split_metrics.to_csv(OUTPUT_DIR / "baseline_split_metrics.csv", index=False)

    metric_columns = [metric for metric, _, _ in SELECTION_METRICS]
    validation_average = (
        split_metrics[split_metrics["split"] == "validation"]
        .groupby("model", observed=True)[metric_columns].mean().reset_index()
    )
    validation_ranking = add_engineering_selection_score(validation_average)
    validation_ranking = validation_ranking.sort_values([
        "engineering_selection_score", "macro_rmse", "model"
    ]).reset_index(drop=True)
    validation_ranking["validation_rank"] = np.arange(1, len(validation_ranking) + 1)
    validation_ranking.to_csv(OUTPUT_DIR / "baseline_validation_ranking.csv", index=False)

    stability = split_metrics.groupby(["model", "split"])[metric_columns].agg(["mean", "std", "min", "max"])
    stability.columns = [f"{metric}_{stat}" for metric, stat in stability.columns]
    stability.reset_index().to_csv(OUTPUT_DIR / "baseline_stability_summary.csv", index=False)
    pd.DataFrame([{
        "feature_set": FEATURE_SET_NAME,
        "features_json": json.dumps(MODEL_FEATURES),
        "manifest_path": str(PATHS.manifest_path),
        "random_seed": RANDOM_SEED,
        "tail_weights_json": json.dumps(TAIL_WEIGHT_LEVELS),
        "final_test_evaluated": False,
    }]).to_csv(OUTPUT_DIR / "baseline_run_contract.csv", index=False)
    display(validation_ranking)
    display(split_metrics)
else:
    print("Baseline training was skipped.")


## Takeaways

`baseline_validation_ranking.csv` is the comparison interface consumed
by the symbolic-regression notebook.  The symbolic search can technically
run without a baseline, but the formal CT3 workflow requires this baseline
to be completed first so formula accuracy can be judged against a known
nonlinear reference under identical splits and metrics.
